In [ ]:
from pathlib import Path

import torch
import torch.optim as optim
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from tqdm import tqdm

from src.utils.baseline import baseline_regression
from src.utils.data import LocalUserScaler, decode_labels, get_windows
from src.utils.feature_engineering import compute_daily_aggregates
from src.utils.metrics import calculate_regression_metrics
from src.utils.rnn_modeling import EarlyStopping, RnnRegressor, get_dataloader


In [2]:
df = get_windows(window_size=7)

In [3]:
agg_rules = {
    "mood": ["count", "mean", "std"],
    "circumplex.arousal": ["count", "mean", "std"],
    "circumplex.valence": ["count", "mean", "std"],
    "activity": ["count", "mean", "std"],
    "call": ["count"],
    "sms": ["count"],
    "screen": ["count", "sum", "mean", "std"],
    "appCat.builtin": ["count", "sum", "mean", "std"],
    "appCat.communication": ["count", "sum", "mean", "std"],
    "appCat.entertainment": ["count", "sum", "mean", "std"],
    "appCat.finance": ["count", "sum", "mean", "std"],
    "appCat.game": ["count", "sum", "mean", "std"],
    "appCat.office": ["count", "sum", "mean", "std"],
    "appCat.other": ["count", "sum", "mean", "std"],
    "appCat.social": ["count", "sum", "mean", "std"],
    "appCat.travel": ["count", "sum", "mean", "std"],
    "appCat.unknown": ["count", "sum", "mean", "std"],
    "appCat.utilities": ["count", "sum", "mean", "std"],
    "appCat.weather": ["count", "sum", "mean", "std"],
}

In [4]:
feature_df = compute_daily_aggregates(df, agg_rules)

train_df = feature_df[feature_df["split"] == "train"]
val_df = feature_df[feature_df["split"] == "val"]
test_df = feature_df[feature_df["split"] == "test"]

In [5]:
features_to_scale = [f"{k}_{agg_func}" for k, v in agg_rules.items() for agg_func in v]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", LocalUserScaler(id_col="id"), ["id"] + features_to_scale),
        ("label", LocalUserScaler(id_col="id"), ["id", "target_mean_mood"]),
        ("user_id", "passthrough", ["id"]),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

pipeline = Pipeline(steps=[("scaler", preprocessor)])
pipeline.set_output(transform="pandas")
pipeline.fit(train_df)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('label', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [6]:
train_df = pipeline.transform(train_df)
val_df = pipeline.transform(val_df)
test_df = pipeline.transform(test_df)

In [7]:
train_dataloader = get_dataloader(
    dataset_kwargs={"df": train_df, "label_col": "target_mean_mood"},
    dataloader_kwargs={"batch_size": 64, "shuffle": True, "num_workers": 1},
)

val_dataloader = get_dataloader(
    dataset_kwargs={"df": val_df, "label_col": "target_mean_mood"},
    dataloader_kwargs={"batch_size": 64, "shuffle": True, "num_workers": 1},
)

test_dataloader = get_dataloader(
    dataset_kwargs={"df": test_df, "label_col": "target_mean_mood"},
    dataloader_kwargs={"batch_size": 64, "shuffle": False, "num_workers": 1},
)

In [8]:
train_metrics = calculate_regression_metrics(
    *baseline_regression(train_df), preprocessor
)
val_metrics = calculate_regression_metrics(*baseline_regression(val_df), preprocessor)
test_metrics = calculate_regression_metrics(*baseline_regression(test_df), preprocessor)

test_metrics

{'Micro': {'MSE': 0.7748, 'RMSE': np.float64(0.8802), 'MAE': 0.6412},
 'Macro': {'MSE': np.float64(0.8059),
  'RMSE': np.float64(0.8439),
  'MAE': np.float64(0.6555)}}

In [9]:
model = RnnRegressor(
    n_features=len(
        train_df.columns.drop(["id", "window_id", "date", "target_mean_mood", "split"])
    ),
    hidden_dim=64,
)

In [10]:
save_path = Path("./best_rnn_model.pt")

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()
stopper = EarlyStopping(patience=3, path=save_path)

In [11]:
epochs = 100
for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0

    pbar = tqdm(train_dataloader, desc=f"Training Epoch {epoch + 1}/{epochs}")
    for x, y, user_idx in pbar:
        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y.view(-1, 1))

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_dataloader)

    # --- VALIDATION ---
    model.eval()
    val_mse = 0
    val_unscaled_mse = 0

    pbar = tqdm(val_dataloader, desc=f"Validation Epoch: {epoch + 1}/{epochs}")
    with torch.no_grad():
        for x, y, u in pbar:
            # outputs = model(x, u)
            outputs = model(x)
            val_mse += criterion(outputs, y.view(-1, 1)).item()

            y_true = y.view(-1).numpy()
            y_pred = outputs.view(-1).numpy()
            user_ids = u

            metrics = calculate_regression_metrics(
                y_true, y_pred, user_ids, preprocessor
            )

            val_unscaled_mse += metrics["Micro"]["MSE"]

    avg_val_loss = val_mse / len(val_dataloader)
    avg_val_rmse = (val_unscaled_mse / len(val_dataloader)) ** 0.5

    print(f">> Epoch {epoch + 1} Results:")
    print(f"   Train MSE: {avg_train_loss:.2f}")
    print(
        f"   Val   MSE: {avg_val_loss:.2f} | RMSE: {avg_val_rmse:.2f} Avg Mood Points"
    )
    print("\n")

    # --- EARLY STOPPING CHECK ---
    if stopper(avg_val_loss, model):
        print(
            f"\nEarly stopping triggered. No improvement for {stopper.patience} epochs."
        )
        model = torch.load(save_path, weights_only=False)
        break

Validation Epoch: 1/100: 100%|██████████| 3/3 [00:00<00:00, 37.99it/s]


>> Epoch 1 Results:
   Train MSE: 1.29
   Val   MSE: 0.96 | RMSE: 0.62 Avg Mood Points




Validation Epoch: 2/100: 100%|██████████| 3/3 [00:00<00:00, 43.11it/s]


>> Epoch 2 Results:
   Train MSE: 1.18
   Val   MSE: 1.25 | RMSE: 0.67 Avg Mood Points




Validation Epoch: 3/100: 100%|██████████| 3/3 [00:00<00:00, 44.58it/s]


>> Epoch 3 Results:
   Train MSE: 1.10
   Val   MSE: 1.01 | RMSE: 0.56 Avg Mood Points




Validation Epoch: 4/100: 100%|██████████| 3/3 [00:00<00:00, 43.97it/s]


>> Epoch 4 Results:
   Train MSE: 1.11
   Val   MSE: 0.99 | RMSE: 0.66 Avg Mood Points



Early stopping triggered. No improvement for 3 epochs.


In [ ]:
model.eval()
with torch.no_grad():
    y_labels = []
    y_preds = []
    user_ids = []
    for x, y, u in tqdm(test_dataloader):
        y_preds.append(model(x).squeeze())
        y_labels.append(y.squeeze())
        user_ids.extend(u)

    y_labels = torch.hstack(y_labels).numpy()
    y_preds = torch.hstack(y_preds).numpy()

    decoded_labels = decode_labels(preprocessor, user_ids, y_labels)
    decoded_predictions = decode_labels(preprocessor, user_ids, y_preds)
    test_metrics = calculate_regression_metrics(
        y_labels, y_preds, user_ids, preprocessor
    )
    print(test_metrics)


100%|██████████| 5/5 [00:00<00:00, 34.71it/s]

{'Micro': {'MSE': 0.3193, 'RMSE': np.float64(0.565), 'MAE': 0.4185}, 'Macro': {'MSE': np.float64(0.3336), 'RMSE': np.float64(0.5465), 'MAE': np.float64(0.4312)}}


In [15]:
from plotly import graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=decoded_labels,
        y=decoded_predictions,
        mode="markers",
        name="Predictions vs True",
    )
)